# Reasoning Effort Levers

The previous notebook asked *which model*. This one asks *how hard should it think* — and
shows that the answer is rarely "as hard as possible".

`reasoning_effort` is a dial with real settings, not a quality slider you pin to maximum.
Turning it up buys accuracy on some tasks, buys nothing on others, and always costs latency
and money. The useful skill is finding **the knee** — the setting past which you are paying
for thinking that no longer changes the answer.

## Learning objectives

1. Say what `reasoning_effort` changes, in tokens rather than adjectives.
2. Sweep a task set across effort levels and measure accuracy, latency and cost per level.
3. Find the knee, and justify a setting with the marginal number rather than a preference.
4. Recognise the tasks where effort is the **wrong lever** and something else is the fix.

## Where this fits

- **Stage:** `01_Foundations/` — no framework, so raw `openai` SDK and no `helpers`.
- **Before:** `01_Reasoning_vs_NonReasoning.ipynb` — whether to use a reasoning model at all.
- **Applied:** `02_Core/05_AI_Agent_Fundamentals/4. Workflow_Pattern/2. Routing/` — choosing
  per request, at runtime.

## Prerequisites and cost

`OPENAI_API_KEY` in the project-root `.env`. The sweep makes roughly
`len(EFFORTS) × len(PROBLEMS)` calls — about 18 by default, a few cents. Raise `TRIALS`
only if you want tighter numbers and are happy to pay for them.

In [1]:
# ============ SETUP ============
import statistics
import time
from dataclasses import dataclass

from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
client = OpenAI()

REASONING_MODEL = "o4-mini"

# Provider-supported settings, cheapest first. Availability varies by model — trim this
# list if your account rejects one rather than editing calls further down.
EFFORTS = ["low", "medium", "high"]

TRIALS = 1  # per (effort, problem). Raise for stabler accuracy, at linear cost.

# USD per 1M tokens, (input, output). Published prices move — set these to today's.
PRICING = {REASONING_MODEL: (1.10, 4.40)}

print(f"model   : {REASONING_MODEL}")
print(f"efforts : {EFFORTS}")

model   : o4-mini
efforts : ['low', 'medium', 'high']


## 1. What the dial actually changes

It does not change the model, the prompt, or the output format. It changes **how many
reasoning tokens the model is willing to spend before committing to an answer**.

Everything else follows from that one fact:

- More reasoning tokens → more time → higher latency.
- Reasoning tokens bill at the **output** rate → higher cost.
- More deliberation → better answers **only if the task had something to deliberate about**.

That last clause is the whole notebook. Effort is not a quality knob; it is a
*deliberation* knob, and deliberation only helps when the failure mode is "didn't think
hard enough" rather than "didn't know" or "was asked badly".

In [2]:
# ============ MEASUREMENT HARNESS ============
@dataclass
class Result:
    effort: str
    problem: str
    answer: str
    correct: bool
    seconds: float
    prompt_tokens: int
    completion_tokens: int
    reasoning_tokens: int

    @property
    def cost_usd(self) -> float:
        inp, out = PRICING[REASONING_MODEL]
        return (self.prompt_tokens / 1e6) * inp + (self.completion_tokens / 1e6) * out


def ask(prompt: str, effort: str) -> tuple[str, float, object]:
    """One call at a given effort. Returns (text, seconds, usage)."""
    started = time.perf_counter()
    resp = client.chat.completions.create(
        model=REASONING_MODEL,
        reasoning_effort=effort,
        messages=[{"role": "user", "content": prompt}],
    )
    elapsed = time.perf_counter() - started
    return (resp.choices[0].message.content or "").strip(), elapsed, resp.usage


def reasoning_of(usage) -> int:
    details = getattr(usage, "completion_tokens_details", None)
    return getattr(details, "reasoning_tokens", 0) or 0

## 2. Why a task *set*, not one question

A single question gives you a coin flip, not a measurement. At `low` the model might
happen to get it right; at `high` it might happen to get it wrong. You would then "discover"
that effort makes no difference, or makes things worse, and both conclusions would be noise.

The set below is small but deliberately mixed: some problems a competent model solves
without deliberating, some genuinely need the working. **Accuracy across a mixed set is the
number worth acting on**, because it is closer to the traffic you would actually serve.

In [3]:
# ============ THE TASK SET ============
# (prompt, accepted-substring) — graded by containment, so answers are constrained to one token.
PROBLEMS: list[tuple[str, str]] = [
    ("A train leaves at 09:40 and the journey takes 2 hours 45 minutes. "
     "What time does it arrive? Answer in HH:MM, 24-hour, nothing else.", "12:25"),

    ("A batch job retries 3 times with exponential backoff starting at 2 seconds "
     "(2, then 4, then 8). Total seconds spent waiting across all retries? Number only.", "14"),

    ("Services deploy one per day Mon-Fri. auth before billing. search the day immediately "
     "after billing. notify on Monday. reporting not on Friday. Which deploys Friday? "
     "One word.", "search"),

    ("A cache holds 4 entries, LRU eviction. Access order: A B C D A E B. "
     "Which key is evicted when E is inserted? One letter.", "c"),

    ("If every alert pages someone, and some alerts are duplicates, does it follow that "
     "some pages are for duplicates? Answer yes or no.", "yes"),

    ("You have 8 identical-looking coins, one heavier. Using a balance scale, what is the "
     "minimum number of weighings that guarantees finding it? Number only.", "2"),
]

print(f"{len(PROBLEMS)} problems x {len(EFFORTS)} efforts x {TRIALS} trial(s) "
      f"= {len(PROBLEMS) * len(EFFORTS) * TRIALS} calls")

6 problems x 3 efforts x 1 trial(s) = 18 calls


## 3. The sweep

One loop, every effort level, same problems, same prompts. The only thing changing is the
dial.

In [4]:
# ============ RUN THE SWEEP ============
results: list[Result] = []

for effort in EFFORTS:
    for prompt, expected in PROBLEMS:
        for _ in range(TRIALS):
            text, secs, usage = ask(prompt, effort)
            results.append(Result(
                effort=effort,
                problem=prompt[:34],
                answer=text,
                correct=expected.lower() in text.lower(),
                seconds=secs,
                prompt_tokens=usage.prompt_tokens,
                completion_tokens=usage.completion_tokens,
                reasoning_tokens=reasoning_of(usage),
            ))
    done = [r for r in results if r.effort == effort]
    print(f"{effort:<8} done — {sum(r.correct for r in done)}/{len(done)} correct")

low      done — 5/6 correct
medium   done — 5/6 correct
high     done — 5/6 correct


In [5]:
# ============ AGGREGATE BY EFFORT ============
def rows():
    for effort in EFFORTS:
        rs = [r for r in results if r.effort == effort]
        yield {
            "effort": effort,
            "accuracy": sum(r.correct for r in rs) / len(rs),
            "sec": statistics.mean(r.seconds for r in rs),
            "reasoning": statistics.mean(r.reasoning_tokens for r in rs),
            "cost": sum(r.cost_usd for r in rs),
        }

summary = list(rows())

print(f"{'effort':<10}{'accuracy':>10}{'mean sec':>10}{'mean reas tok':>15}{'total $':>10}")
print("-" * 55)
for s in summary:
    print(f"{s['effort']:<10}{s['accuracy']:>9.0%}{s['sec']:>10.2f}"
          f"{s['reasoning']:>15.0f}{s['cost']:>10.4f}")

effort      accuracy  mean sec  mean reas tok   total $
-------------------------------------------------------
low             83%      2.53            245    0.0073
medium          83%      3.19            331    0.0095
high            83%      6.68           1003    0.0272


### Discussion of the output

Read the **mean reasoning tokens** column first — that is the dial doing its job, and it
should climb clearly across the levels. If it does not, the parameter is not reaching the
model and nothing else in this notebook means anything.

Then read accuracy against it. The usual shape is a **step then a plateau**: a real gain
from `low` to `medium`, and much less from `medium` to `high`, while cost and latency keep
climbing linearly. That plateau is the point.

If accuracy is flat at 100% across all three, your task set is too easy to show the effect —
add harder problems rather than concluding effort does nothing.

## 4. Finding the knee

"Is high better than medium" is the wrong question, because on a hard enough set the answer
is almost always a technical yes. The question that decides a production setting is
**how much did the last step cost per point of accuracy it bought.**

In [6]:
# ============ MARGINAL COST OF EACH STEP UP ============
print(f"{'step':<20}{'+accuracy':>11}{'+sec':>9}{'+cost $':>10}{'$ per +1pp':>13}")
print("-" * 63)
for prev, nxt in zip(summary, summary[1:]):
    d_acc = nxt["accuracy"] - prev["accuracy"]
    d_sec = nxt["sec"] - prev["sec"]
    d_cost = nxt["cost"] - prev["cost"]
    per_pp = (d_cost / (d_acc * 100)) if d_acc > 0 else float("inf")
    per_pp_s = f"{per_pp:.4f}" if d_acc > 0 else "no gain"
    print(f"{prev['effort'] + ' -> ' + nxt['effort']:<20}{d_acc:>10.0%}"
          f"{d_sec:>9.2f}{d_cost:>10.4f}{per_pp_s:>13}")

print("\nThe knee is the last step where '$ per +1pp' is a number you would actually pay.")
print("Once a step reads 'no gain', every further step is pure cost.")

step                  +accuracy     +sec   +cost $   $ per +1pp
---------------------------------------------------------------
low -> medium               0%     0.66    0.0023      no gain
medium -> high              0%     3.49    0.0177      no gain

The knee is the last step where '$ per +1pp' is a number you would actually pay.
Once a step reads 'no gain', every further step is pure cost.


In [ ]:
# ============ OPTIONAL PLOT ============
# matplotlib lives in the `data` extra (uv pip install -e ".[data]"), so this is guarded —
# the tables above already carry the argument.
try:
    import matplotlib.pyplot as plt

    fig, ax1 = plt.subplots(figsize=(7, 4))
    x = [s["effort"] for s in summary]
    ax1.plot(x, [s["accuracy"] for s in summary], marker="o", label="accuracy")
    ax1.set_ylabel("accuracy")
    ax1.set_ylim(0, 1.05)

    ax2 = ax1.twinx()
    ax2.plot(x, [s["sec"] for s in summary], marker="s", linestyle="--", color="tab:red",
             label="mean latency (s)")
    ax2.set_ylabel("mean latency (s)")

    ax1.set_xlabel("reasoning_effort")
    ax1.set_title("Accuracy plateaus; latency does not")
    fig.legend(loc="lower right", bbox_to_anchor=(0.9, 0.15))
    plt.tight_layout()
    plt.show()
except ImportError:
    print("matplotlib not installed — skipping the plot. Install with: uv pip install -e \".[data]\"")

## 5. When effort is the wrong lever

Here is the failure this notebook most wants you to avoid: reaching for more effort when the
problem is not a thinking problem.

The task below cannot be solved by deliberating, because the information needed is simply
not present. A model at `high` will spend many more reasoning tokens arriving at the same
place a model at `low` reached immediately — the bill goes up, the answer does not improve.

In [ ]:
# ============ A TASK NO AMOUNT OF THINKING FIXES ============
UNANSWERABLE = (
    "What was the total refund value processed by our EU billing service last Tuesday? "
    "If the data needed is not provided, reply exactly: INSUFFICIENT DATA"
)

print(f"{'effort':<10}{'sec':>8}{'reasoning tok':>15}{'$':>10}  answer")
print("-" * 62)
for effort in EFFORTS:
    text, secs, usage = ask(UNANSWERABLE, effort)
    r = Result(effort, "unanswerable", text, "INSUFFICIENT" in text.upper(), secs,
               usage.prompt_tokens, usage.completion_tokens, reasoning_of(usage))
    print(f"{effort:<10}{r.seconds:>8.2f}{r.reasoning_tokens:>15}{r.cost_usd:>10.5f}  "
          f"{text[:30]}")

### Discussion of the output

Every level should reach the same conclusion, and the higher levels should pay noticeably
more reasoning tokens to get there.

That is the signature of a **misapplied lever**. When you see effort climbing with no
accuracy movement, the fix is upstream, not on the dial:

| What is actually wrong | The real fix |
|---|---|
| The model lacks a fact | Retrieval, a tool call, or a database — not thinking |
| The question is ambiguous | A clearer prompt or a required output schema |
| The task is mechanical | A cheaper non-reasoning model, or plain code |
| The task is genuinely hard | *Now* effort is the right lever |

Only the last row is an effort problem. Reaching for the dial in the first three rows is how
a bill grows without a metric moving.

## Key takeaways

1. **`reasoning_effort` buys deliberation, not quality.** It only converts to quality when
   the failure mode was insufficient thinking.
2. **Measure on a set, never one question.** A single prompt gives you a coin flip and an
   confident wrong conclusion.
3. **Accuracy plateaus; cost and latency do not.** The shape is usually a step from `low` to
   `medium` and much less beyond — so the default should rarely be `high`.
4. **Justify the setting with the marginal number** — cost per extra accuracy point of the
   last step up — rather than with a preference for the strongest setting.
5. **Flat accuracy with rising reasoning tokens means you picked the wrong lever.** Look for
   a missing fact, an ambiguous prompt, or a task that never needed a reasoning model.
6. **Reasoning tokens are billed and invisible**, so the cost of turning the dial up is
   larger than the visible output suggests.

### Next

- `02_Core/05_AI_Agent_Fundamentals/4. Workflow_Pattern/2. Routing/` — stop choosing one
  setting for all traffic and let a cheap classifier decide per request.